# Audio Compliance Scanner

Fully Google Colab compatible. Upload files through Colab buttons, run the analysis, and get a JSON result.

Inputs:
- Required: one target audio/video file to scan.
- Optional: one or more reference audio/video files for acoustic copyright matching.

Accepted formats include `.mp4`, `.mov`, `.mkv`, `.webm`, `.wav`, `.mp3`, `.m4a`, `.aac`, `.flac`, and `.ogg`.

In [8]:
# Run this cell first. Installs happen inside the temporary Colab runtime, not on your computer.
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install numpy scipy


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [9]:
from google.colab import files

print('Upload the TARGET media file you want to scan.')
print('If you skip this, the notebook will automatically generate a synthetic compliance test failure.')
target_uploaded = files.upload()
TARGET_PATH = next(iter(target_uploaded.keys()), None)

print('\nOptional: upload REFERENCE file(s) for copyright/acoustic matching.')
print('If you only want EAS tone detection, cancel or upload nothing.')
reference_uploaded = files.upload() if TARGET_PATH else {}
REFERENCE_PATHS = list(reference_uploaded.keys())

print('\nTarget Ingest Path:', TARGET_PATH)
print('Reference Ingest Database:', REFERENCE_PATHS if REFERENCE_PATHS else 'None')

Upload the TARGET media file you want to scan.
If you skip this, the notebook will automatically generate a synthetic compliance test failure.


Saving test_unauthorized_music_use.wav to test_unauthorized_music_use (2).wav

Optional: upload REFERENCE file(s) for copyright/acoustic matching.
If you only want EAS tone detection, cancel or upload nothing.


Saving test_unauthorized_music_use (1).wav to test_unauthorized_music_use (1) (2).wav

Target Ingest Path: test_unauthorized_music_use (2).wav
Reference Ingest Database: ['test_unauthorized_music_use (1) (2).wav']


In [11]:
import hashlib
import json
import subprocess
import tempfile
import time
from collections import Counter
from pathlib import Path

import numpy as np
from scipy.io import wavfile
from scipy.ndimage import maximum_filter
from scipy.signal import stft

SAMPLE_RATE = 22050

def extract_mono_audio(media_path, sample_rate=SAMPLE_RATE):
    media_path = Path(media_path)
    wav_path = Path(tempfile.mkdtemp()) / f'{media_path.stem}_mono.wav'
    command = [
        'ffmpeg', '-y', '-i', str(media_path),
        '-vn', '-ac', '1', '-ar', str(sample_rate), '-sample_fmt', 's16',
        str(wav_path)
    ]
    try:
        subprocess.run(command, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
    except subprocess.CalledProcessError as exc:
        stderr = exc.stderr.decode('utf-8', errors='replace') if exc.stderr else ''
        raise RuntimeError(f'FFmpeg could not read {media_path.name}. Details: {stderr[:1200]}')

    sr, data = wavfile.read(wav_path)
    if data.ndim > 1:
        data = data.mean(axis=1)
    data = data.astype(np.float32) / 32768.0
    return sr, np.nan_to_num(data)

def goertzel_power(window, sample_rate, target_hz):
    n = len(window)
    k = int(0.5 + (n * target_hz / sample_rate))
    omega = (2.0 * np.pi * k) / n
    coeff = 2.0 * np.cos(omega)
    q0 = 0.0
    q1 = 0.0
    q2 = 0.0
    for sample in window:
        q0 = coeff * q1 - q2 + float(sample)
        q2 = q1
        q1 = q0
    return float((q1 * q1 + q2 * q2 - coeff * q1 * q2) / (n * n))

def detect_eas_tone(audio_data, sample_rate, threshold=0.18, min_duration_sec=2.0):
    window_size = int(0.5 * sample_rate)
    hop_size = int(0.1 * sample_rate)
    required_hits = int(np.ceil(min_duration_sec / 0.1))

    if len(audio_data) < window_size:
        return {'eas_tone_present': False, 'confidence_score': 0.0, 'sustained_seconds': 0.0}

    best_score = 0.0
    current_hits = 0
    best_hits = 0

    for start in range(0, len(audio_data) - window_size + 1, hop_size):
        window = audio_data[start:start + window_size] * np.hanning(window_size)
        broadband_power = float(np.mean(window * window)) + 1e-12

        # Track energy across all valid regulatory warning frequencies
        power_1050 = goertzel_power(window, sample_rate, 1050.0)
        power_853 = goertzel_power(window, sample_rate, 853.0)
        power_960 = goertzel_power(window, sample_rate, 960.0)

        # Compute relative spectral scores
        score_1050 = power_1050 / broadband_power
        score_dual_tone = (power_853 + power_960) / broadband_power

        # Take the maximum score present in this frame window
        frame_score = max(score_1050, score_dual_tone)
        best_score = max(best_score, frame_score)

        if frame_score >= threshold:
            current_hits += 1
            best_hits = max(best_hits, current_hits)
        else:
            current_hits = 0

    return {
        'eas_tone_present': bool(best_hits >= required_hits),
        'confidence_score': round(float(min(1.0, best_score / threshold)), 3),
        'sustained_seconds': round(float(best_hits * 0.1), 2)
    }

def generate_fingerprints(audio_data, sample_rate):
    if len(audio_data) == 0:
        return []

    peak = float(np.max(np.abs(audio_data)))
    if peak > 0:
        audio_data = audio_data / peak

    freqs, times, spectrum = stft(
        audio_data,
        fs=sample_rate,
        nperseg=2048,
        noverlap=1536,
        boundary=None
    )
    magnitude = np.log1p(np.abs(spectrum))
    local_peaks = maximum_filter(magnitude, size=15) == magnitude
    threshold = np.percentile(magnitude, 92)
    peak_points = np.argwhere(local_peaks & (magnitude >= threshold))
    points = sorted((int(t), float(times[t]), float(freqs[f])) for f, t in peak_points)

    hashes = []
    for anchor_idx, (time_frame, absolute_time, freq_1) in enumerate(points):
        for target_time_frame, _, freq_2 in points[anchor_idx + 1:anchor_idx + 13]:
            delta_frames = target_time_frame - time_frame
            if 1 <= delta_frames <= 40:
                raw = f'{int(freq_1 // 25)}|{int(freq_2 // 25)}|{delta_frames}'.encode('utf-8')
                hash_key = hashlib.sha1(raw).hexdigest()[:20]
                hashes.append((hash_key, round(absolute_time, 3)))
    return hashes

def build_reference_index(reference_paths):
    index = {}
    for reference_path in reference_paths:
        sr, audio = extract_mono_audio(reference_path)
        asset_id = Path(reference_path).stem
        for hash_key, offset in generate_fingerprints(audio, sr):
            index.setdefault(hash_key, []).append((asset_id, offset))
    return index

def match_fingerprint(audio_data, sample_rate, reference_index, min_votes=18):
    if not reference_index:
        return {
            'match_detected': False,
            'reference_asset_id': None,
            'match_timestamp_start': None,
            'confidence_score': 0.0,
            'matching_hash_votes': 0
        }

    query_hashes = generate_fingerprints(audio_data, sample_rate)
    votes = Counter()
    for query_hash, query_offset in query_hashes:
        for asset_id, reference_offset in reference_index.get(query_hash, []):
            offset_delta = round(reference_offset - query_offset, 1)
            votes[(asset_id, offset_delta)] += 1

    if not votes:
        return {
            'match_detected': False,
            'reference_asset_id': None,
            'match_timestamp_start': None,
            'confidence_score': 0.0,
            'matching_hash_votes': 0
        }

    (asset_id, alignment), vote_count = votes.most_common(1)[0]
    confidence = min(1.0, vote_count / max(min_votes * 2, len(query_hashes) * 0.02, 1))
    match_detected = vote_count >= min_votes
    return {
        'match_detected': bool(match_detected),
        'reference_asset_id': asset_id if match_detected else None,
        'match_timestamp_start': max(0.0, round(float(-alignment), 3)) if match_detected else None,
        'confidence_score': round(float(confidence), 3),
        'matching_hash_votes': int(vote_count)
    }

def create_synthetic_test_files(sample_rate=SAMPLE_RATE):
    duration_sec = 4
    t = np.arange(sample_rate * duration_sec) / sample_rate

    reference = (
        0.35 * np.sin(2 * np.pi * 440 * t) +
        0.22 * np.sin(2 * np.pi * 880 * t) +
        0.14 * np.sin(2 * np.pi * 1320 * t)
    ).astype(np.float32)
    eas_tone = (0.35 * np.sin(2 * np.pi * 1050 * t)).astype(np.float32)
    target = np.clip(reference + eas_tone, -0.95, 0.95).astype(np.float32)

    wavfile.write('synthetic_reference.wav', sample_rate, np.int16(reference * 32767))
    wavfile.write('synthetic_anomaly_target.wav', sample_rate, np.int16(target * 32767))
    return 'synthetic_anomaly_target.wav', ['synthetic_reference.wav']

def analyze_audio(target_path, reference_paths=None):
    reference_paths = reference_paths or []
    if target_path is None:
        target_path, reference_paths = create_synthetic_test_files()
        print('No upload found. Generated synthetic_anomaly_target.wav and synthetic_reference.wav inside Colab.')

    start_time = time.perf_counter()

    sample_rate, target_audio = extract_mono_audio(target_path)
    eas_alert = detect_eas_tone(target_audio, sample_rate)

    reference_index = build_reference_index(reference_paths) if reference_paths else {}
    audio_match = match_fingerprint(
        target_audio,
        sample_rate,
        reference_index
    )

    processing_time_ms = round((time.perf_counter() - start_time) * 1000, 1)
    anomaly = bool(eas_alert['eas_tone_present'] or audio_match['match_detected'])

    return {
        'status': 'anomaly_flagged' if anomaly else 'success',
        'scanned_file': str(target_path),
        'audio_match': audio_match,
        'eas_alert': eas_alert,
        'metrics': {'processing_time_ms': processing_time_ms}
    }

In [12]:
# Execute the compliance tracking pipeline
result = analyze_audio(TARGET_PATH, REFERENCE_PATHS)

# Extract direct explicit True/False states for the final audit logging
eas_detected = result['eas_alert']['eas_tone_present']
unauthorized_music_matched = result['audio_match']['match_detected']

print("=" * 60)
print("                    COMPLIANCE REPORT                  ")
print("=" * 60)
print(f"Scanned Asset File:          {result['scanned_file']}")
print(f"EAS Alert Detected:          {eas_detected}")
print(f"Unauthorized Music Matched:  {unauthorized_music_matched}")
print("-" * 60)
print(f"Pipeline Runtime Latency:    {result['metrics']['processing_time_ms']} ms")
print("=" * 60)

# Save the comprehensive JSON artifact locally
with open('audio_compliance_result.json', 'w') as f:
    json.dump(result, f, indent=2)

# Auto-download back to host system
files.download('audio_compliance_result.json')

                    COMPLIANCE REPORT                  
Scanned Asset File:          test_unauthorized_music_use (2).wav
EAS Alert Detected:          False
Unauthorized Music Matched:  True
------------------------------------------------------------
Pipeline Runtime Latency:    738.1 ms


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>